# 🦜LangChain `create_agent`

**Welcome!**
In the previous notebook, we built a graph manually (nodes, edges, state) using LangGraph. That gave us total control but required a lot of setup.
This notebook introduces **`create_agent`**, a high-level function from the `langchain` module.

###  What is `create_agent`?
It is a **pre-assembled LangGraph**.
Under the hood, `create_agent` automatically:
1.  Creates a `StateGraph`.
2.  Adds a model node and a tool node.
3.  Connects them with the correct edges (loops).
4.  Compiles it into a `CompiledGraph`.

### **📚 What We Will Cover**
1.  **`create_agent`:** Instantiating a pre-built ReAct agent workflow (Model + Tools loop) without needing to define nodes and edges manually.
2.  **Standard Interrupts:** Handling missing info (just like before).
3.  **Middleware:** A way to hook into the agent's brain to modify its behavior (e.g., adding a "Human Approval" step before dangerous tools are run).

![Diagram](./images/10_img.png)

In [ ]:
# Import the building blocks. create_agent supplies the reasoning/tool loop built manually in notebook 1.
# Load API configuration from .env; no API key should be written directly into notebook code.

from langchain.agents import create_agent
from langraph_prompts import ACTION_PROMPT_TEMPLATE
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt
from datetime import datetime
from dotenv import load_dotenv

load_dotenv(override=True) # .env values take precedence over existing shell env vars

In [ ]:
# Define the leave tools and a tool that can pause to ask a person a question.
# These demo tools return sample results; they do not update an HR database.

# @tool is a decorator: it wraps the following function as a tool whose name, inputs, and description the model can see.
@tool
# def defines a reusable function. user_id: str expects text; -> int describes an integer return value.
def get_time_off_balance(user_id: str) -> int:
    '''
    Get the time off balance for a user. 

    Args :
        user_id: A unique string representing the user id

    Returns:
        An integer value representing the time-off balance
    '''
    # Fixed sample balance for every employee; this does not read a real account.
    return 10

# @tool is a decorator: it wraps the following function as a tool whose name, inputs, and description the model can see.
@tool
# Inputs are employee ID and date strings. -> dict describes a dictionary: named keys paired with values.
def process_time_off(user_id: str, start_date: str, end_date: str) -> dict:
    '''
    Process time-off request for a user

    Args :
        user_id: A unique string representing the user id
        start_date: A date string in YYYY-MM-DD format for the start date of the time-off request
        end_date: A date string in YYYY-MM-DD format for the end date of the time-off request

    Returns:
        A dictionary with the following keys:
        - status: A string value representing the status of the time-off request
        - message: A string value representing the message of the time-off request
    '''
    # Parse date text into a date/time object. %Y-%m-%d means four-digit year, month, day.
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    # Parse date text into a date/time object. %Y-%m-%d means four-digit year, month, day.
    end_date = datetime.strptime(end_date, "%Y-%m-%d")

    # Subtracting dates produces a time difference; .days reads its whole-day count.
    days_requested = end_date - start_date
    # .invoke runs the wrapped tool, supplying its named input in a dictionary.
    time_off_balance = get_time_off_balance.invoke({"user_id": user_id})

    # Start an empty dictionary; later assignments fill in status and message.
    result = {}

    # if selects a branch. Add 1 to count both the start date and end date as leave days.
    if days_requested.days + 1 > time_off_balance:
        result['status'] = 'error'
        # An f-string inserts values from {...} into the displayed text.
        result['message'] = f'Time off request failed. you only have {time_off_balance} days of remaining leaves but requested {days_requested.days} days'

    else:
        # -= subtracts from this local variable only; the new balance is not saved for future tool calls.
        time_off_balance -= days_requested.days + 1
        result['status'] = 'success'
        # An f-string inserts values from {...} into the displayed text.
        result['message'] = f'Time off request processed successfully for {days_requested.days + 1} days. Your remaining time off balance is {time_off_balance}'


    return result


# @tool is a decorator: it wraps the following function as a tool whose name, inputs, and description the model can see.
@tool
# This tool asks the person a question instead of guessing missing information.
def get_additional_info_from_human( message: str) -> str:
    '''
        Raises an interrupt to fetch additional information from the human requesting the action

        Args:
            message: a message string from the AI requesting the user for missing information

        Returns:
            A string value representing the response with additional information from the human

    '''
    # Pause and expose this question to the caller. On resume the node restarts; this call receives the supplied resume value.
    interrupt_result = interrupt(message)

    # Read the user_message entry from the dictionary supplied through Command(resume=...).
    return interrupt_result['user_message']


## 1) Creating the Agent (The Easy Way)

* **Manual Graph (Previous Notebook):** Like building a PC from scratch. You buy the CPU, RAM, and Motherboard, and connect the wires yourself. It's powerful but takes time.
* **`create_agent` (This Notebook):** Like buying a pre-built laptop. You just turn it on. It has the same components inside (LLM, Tools, Memory), but LangChain assembled them for you.

###  Code Context
In the code below:
* `create_agent(...)`: This single function replaces the `StateGraph`, `add_node`, `add_edge`, and `compile` steps.
* It automatically creates a graph where the LLM decides to call tools and loops back until it's done.
* `checkpointer=InMemorySaver()`: We still need this to enable "Pause/Resume" functionality.

📚 **[Docs: create_agent](https://docs.langchain.com/oss/python/langchain/agents)**

![Diagram](./images/11_img.png)

In [ ]:
# Combine the model, tools, instructions, and conversation memory into a ready-to-run agent.

user_id = '1' 
# Prepare the model client. temperature=0 requests less randomness; it does not guarantee identical answers.
llm = init_chat_model(model="gpt-4o", temperature=0)
# Square brackets create a list: these are the tools available to this agent.
tools = [get_time_off_balance, process_time_off, get_additional_info_from_human]

todays_date = datetime.now().strftime("%Y-%m-%d")

# .format fills named placeholders in the instruction template with actual values.
system_prompt = ACTION_PROMPT_TEMPLATE.format(todays_date=todays_date, user_id=user_id)

# create_agent builds the model/tool loop and attaches the supplied instructions and optional checkpoint store.
agent = create_agent(llm, tools=tools, system_prompt=system_prompt, checkpointer=InMemorySaver())

In [ ]:
# Choose a request and a thread ID. Reuse this config when continuing the same conversation.

user_message = "I want to take leave"

# Nested dictionaries configure the thread ID, the key used to find this conversation checkpoint.
config = {'configurable': {'thread_id': user_id}}

In [ ]:
# invoke runs the agent until it finishes or pauses. This makes model API calls.
# The nested dictionary supplies a list containing one user message.

result = agent.invoke({"messages": [{"role": "user", "content": user_message}]}, config=config)

result

## 1.1) Resuming the Agent

Just like in the previous notebook, our `get_additional_info_from_human` tool triggered an **Interrupt**. The agent is now paused, waiting for the "End Date".
We use the same `Command(resume=...)` pattern to provide the answer and un-pause execution.

### Code Context
In the code below:
* `agent.invoke(...)`: We pass the `Command` object with the user's answer (`"tomorrow"`).
* The agent takes this answer, pretends the tool returned it, and continues its logic to book the leave.

In [ ]:
# Supply the answer to the missing-information question from the previous run.
# Run this only when that agent is paused on get_additional_info_from_human.

user_message = "tomorrow"

result = agent.invoke(Command(resume={'user_message': user_message}), config=config)

result

## 2) Middleware (Human-in-the-Loop)

Middleware in LangChain is a new abstraction that lets you hook into and control the agent’s core loop at defined points (before the model is called, as the model is called, and after the model/tool step finishes). It is conceptually similar to HTTP middleware in web frameworks: you can stack multiple middleware components that each inspect or modify state as a request “flows” through the agent.

Each middleware can read and modify the agent state, messages, tools, or model configuration at specific hook points, without changing the agent implementation itself.

In our code, we are adding this "Are you sure?" popup to the **Book Leave** tool to prevent accidental bookings.

### Code Context
In the code below:
* `HumanInTheLoopMiddleware`: This is the component that intercepts tool calls.
* `interrupt_on`: This dictionary defines **which** tools trigger the pause. We map `process_time_off` to specific permissions, ensuring the agent cannot run this specific tool without external approval.

📚 **[Docs: Middleware](https://docs.langchain.com/oss/python/langchain/middleware/overview)**

![Diagram](./images/12_img.png)

In [ ]:
# Middleware adds behavior around agent execution; here it introduces human review of a tool call.

from langchain.agents.middleware import HumanInTheLoopMiddleware

In [ ]:
# Create another agent that pauses before process_time_off so a human can approve or reject it.
# This agent has its own in-memory checkpoint store.

agent_with_middleware = create_agent(llm, 
    tools=tools, 
    system_prompt=system_prompt, 
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "process_time_off": {
                    "allowed_decisions": ["approve", "reject"],
                },
            }
        ),
    ],
    )

In [ ]:
# Set up a new example request for the agent with human approval.

user_message = "I want to take leave on 17-06-2026"
# Nested dictionaries configure the thread ID, the key used to find this conversation checkpoint.
config = {'configurable': {'thread_id': user_id}}

In [ ]:
# Start the approval example. Inspect its result to see whether it asks for details or for a decision.

agent_with_middleware.invoke({"messages": [{"role": "user", "content": user_message}]}, config=config)

In [ ]:
# Answer a missing-information interrupt with the format expected by the human-question tool.
# Use this cell only if the current pause asks for missing details.

agent_with_middleware.invoke(Command(resume={'user_message': 'single day'}), config=config)

In [ ]:
# Approve the pending tool call using the middleware decision format.
# Use this cell only when the current pause is an approval request; it allows that tool to execute.

agent_with_middleware.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=config)      